In [2]:
import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.models import load_model
from collections import deque

In [3]:
model = load_model('sign_language_lstm_improved_gaussian.h5')
actions = np.array(['good', 'hello', 'iloveyou', 'please', 'thankyou', 'yes', 'you'])

In [4]:
sequence = []
sentence = []
threshold = 0.85

# Smoothing buffers
prob_buffer = deque(maxlen=5)
pred_buffer = deque(maxlen=10)

In [5]:
mp_holistic = mp.solutions.holistic
mp_drawing = mp.solutions.drawing_utils

In [6]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results

In [7]:
def extract_keypoints(results):
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() \
        if results.left_hand_landmarks else np.zeros(21 * 3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() \
        if results.right_hand_landmarks else np.zeros(21 * 3)
    return np.concatenate([lh, rh])

In [8]:
def draw_styled_landmarks(image, results):
    mp_drawing.draw_landmarks(
        image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(121, 44, 250), thickness=2, circle_radius=2)
    )
    mp_drawing.draw_landmarks(
        image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2)
    )

In [9]:
def draw_probability_bars(image, avg_res):
    for num, (action, prob) in enumerate(zip(actions, avg_res)):
        cv2.rectangle(image, (640, 60 + num * 40), (760, 90 + num * 40), (200, 200, 200), -1)
        cv2.rectangle(image, (640, 60 + num * 40), (640 + int(prob * 120), 90 + num * 40), (245, 117, 16), -1)
        cv2.putText(image, action, (645, 83 + num * 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
    return image


In [10]:
def draw_controls_info(image):
    """Draw keyboard shortcut info on screen"""
    # Background box for controls
    cv2.rectangle(image, (0, 390), (380, 490), (50, 50, 50), -1)
    cv2.rectangle(image, (0, 390), (380, 490), (255, 255, 255), 1)

    cv2.putText(image, 'CONTROLS:', (10, 410),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 1, cv2.LINE_AA)
    cv2.putText(image, 'C  - Delete last word', (10, 435),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 150), 1, cv2.LINE_AA)
    cv2.putText(image, 'L  - Clear all words', (10, 458),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 150, 255), 1, cv2.LINE_AA)
    cv2.putText(image, 'Q  - Quit', (10, 481),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 255), 1, cv2.LINE_AA)
    return image

In [11]:
flash_message = ''
flash_timer = 0

def draw_flash_message(image, message):
    """Show a temporary feedback message in center of screen"""
    if message:
        cv2.putText(image, message, (180, 250),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3, cv2.LINE_AA)
    return image


In [12]:
cap = cv2.VideoCapture(0)
cv2.namedWindow('Real-time Sign Language Translation')

with mp_holistic.Holistic(
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5) as holistic:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Make detections
        image, results = mediapipe_detection(frame, holistic)

        # Draw styled landmarks
        draw_styled_landmarks(image, results)

        # Extract keypoints
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]

        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0), verbose=0)[0]

            # Smoothing
            prob_buffer.append(res)
            avg_res = np.mean(prob_buffer, axis=0)

            pred_index = np.argmax(avg_res)
            pred_buffer.append(pred_index)

            consistent = pred_buffer.count(pred_index) >= 8

            print(f"Predicted: {actions[pred_index]} | Avg Conf: {avg_res[pred_index]:.2f} | Consistent: {consistent}")

            # Thresholding + Sentence Update
            if avg_res[pred_index] > threshold and consistent:
                if len(sentence) > 0:
                    if actions[pred_index] != sentence[-1]:
                        sentence.append(actions[pred_index])
                else:
                    sentence.append(actions[pred_index])

            if len(sentence) > 5:
                sentence = sentence[-5:]

            # Draw probability bars
            draw_probability_bars(image, avg_res)

        # --- Top banner with sentence ---
        cv2.rectangle(image, (0, 0), (640, 40), (245, 117, 16), -1)
        cv2.putText(image, ' '.join(sentence), (3, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

        # --- Draw controls info box ---
        draw_controls_info(image)

        # --- Draw flash message if active ---
        if flash_timer > 0:
            draw_flash_message(image, flash_message)
            flash_timer -= 1

        cv2.imshow('Real-time Sign Language Translation', image)

        # --- Keyboard Input ---
        key = cv2.waitKey(10) & 0xFF

        if key == ord('q'):
            # Quit
            break

        elif key == ord('c'):
            # Delete last word one by one
            if len(sentence) > 0:
                removed = sentence.pop()
                flash_message = f'Removed: {removed}'
                flash_timer = 30  # show message for ~30 frames
                print(f">>> Removed last word: {removed}")
            else:
                flash_message = 'Nothing to remove!'
                flash_timer = 30
                print(">>> Sentence is already empty!")

        elif key == ord('l'):
            # Clear everything
            sentence.clear()
            sequence.clear()
            prob_buffer.clear()
            pred_buffer.clear()
            flash_message = 'Cleared All!'
            flash_timer = 30
            print(">>> Cleared all!")

cap.release()
cv2.destroyAllWindows()

Predicted: hello | Avg Conf: 0.69 | Consistent: False
Predicted: hello | Avg Conf: 0.76 | Consistent: False
Predicted: hello | Avg Conf: 0.80 | Consistent: False
Predicted: hello | Avg Conf: 0.84 | Consistent: False
Predicted: hello | Avg Conf: 0.86 | Consistent: False
Predicted: hello | Avg Conf: 0.91 | Consistent: False
Predicted: hello | Avg Conf: 0.95 | Consistent: False
Predicted: hello | Avg Conf: 0.96 | Consistent: True
Predicted: hello | Avg Conf: 0.97 | Consistent: True
Predicted: hello | Avg Conf: 0.98 | Consistent: True
Predicted: hello | Avg Conf: 0.99 | Consistent: True
Predicted: hello | Avg Conf: 0.99 | Consistent: True
Predicted: hello | Avg Conf: 0.99 | Consistent: True
Predicted: hello | Avg Conf: 0.99 | Consistent: True
Predicted: hello | Avg Conf: 0.99 | Consistent: True
Predicted: hello | Avg Conf: 0.99 | Consistent: True
Predicted: hello | Avg Conf: 0.99 | Consistent: True
Predicted: hello | Avg Conf: 0.99 | Consistent: True
Predicted: hello | Avg Conf: 0.99 | Con